## creating token embeddings

In [2]:
import torch
input_ids = torch.tensor([2, 3, 5, 1])

In [4]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [5]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [6]:
print(embedding_layer(torch.tensor([3]))) # a embedding vector for token ID 3

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [7]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## positional embedding (encoding word positions)

In [15]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [16]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True,num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [8]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [18]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read() 

In [21]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("token IDs:\n" , inputs)
print("\ninputs shape:\n" , inputs.shape)

token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

inputs shape:
 torch.Size([8, 4])


In [22]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [10]:

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [11]:
print(pos_embedding_layer.weight)

Parameter containing:
tensor([[ 0.7504,  0.7353,  1.7375,  ..., -0.2488, -0.7666, -0.2277],
        [ 0.5131, -0.2599,  1.6423,  ..., -0.0586,  1.3007,  0.4118],
        [ 0.1582, -1.1788, -0.4651,  ...,  0.8060, -0.0383,  1.4335],
        [-1.1071, -0.9817, -0.6754,  ..., -0.5813, -0.3207,  0.8139]],
       requires_grad=True)


In [12]:
pos_embedding = pos_embedding_layer(torch.arange(max_length))
print(pos_embedding.shape)
print(pos_embedding)

torch.Size([4, 256])
tensor([[ 0.7504,  0.7353,  1.7375,  ..., -0.2488, -0.7666, -0.2277],
        [ 0.5131, -0.2599,  1.6423,  ..., -0.0586,  1.3007,  0.4118],
        [ 0.1582, -1.1788, -0.4651,  ...,  0.8060, -0.0383,  1.4335],
        [-1.1071, -0.9817, -0.6754,  ..., -0.5813, -0.3207,  0.8139]],
       grad_fn=<EmbeddingBackward0>)


In [25]:
input_embeddings = token_embeddings + pos_embedding
print(input_embeddings.shape)

torch.Size([8, 4, 256])
